In [1]:
import os
import numpy as np
import pandas as pd

data_dir = "/kaggle/input/datasets/rpghost/scaffoldsplit"

train_df = pd.read_csv(f"{data_dir}/train.csv")
val_df   = pd.read_csv(f"{data_dir}/val.csv")
test_df  = pd.read_csv(f"{data_dir}/test.csv")

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(name, df.shape)
    print(df["hERG_blocker"].value_counts().to_dict())
    assert df["smiles"].notna().all()
    assert df["hERG_blocker"].notna().all()
    assert set(df["hERG_blocker"].unique()).issubset({0, 1})

Train (7451, 4)
{1: 4932, 0: 2519}
Validation (954, 4)
{1: 663, 0: 291}
Test (1944, 4)
{1: 1329, 0: 615}


In [2]:
!pip install rdkit xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 56.6 MB/s eta 0:00:00:00:0100:01


In [3]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from xgboost import XGBClassifier

print("RDKit and XGBoost imported successfully")

RDKit and XGBoost imported successfully


In [4]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from tqdm.auto import tqdm
import numpy as np

# Create the ECFP4 fingerprint generator once
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def smiles_to_fingerprints(smiles_list):
    fingerprints = []

    for smiles in tqdm(smiles_list):
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            raise ValueError(f"Invalid SMILES encountered: {smiles}")

        fp = morgan_generator.GetFingerprint(mol)

        arr = np.zeros(2048, dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        fingerprints.append(arr)

    return np.asarray(fingerprints, dtype=np.uint8)


X_train = smiles_to_fingerprints(train_df["smiles"].tolist())
X_val = smiles_to_fingerprints(val_df["smiles"].tolist())
X_test = smiles_to_fingerprints(test_df["smiles"].tolist())

y_train = train_df["hERG_blocker"].astype(int).to_numpy()
y_val = val_df["hERG_blocker"].astype(int).to_numpy()
y_test = test_df["hERG_blocker"].astype(int).to_numpy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

  0%|          | 0/7451 [00:00<?, ?it/s]

  0%|          | 0/954 [00:00<?, ?it/s]

  0%|          | 0/1944 [00:00<?, ?it/s]

Train: (7451, 2048)
Validation: (954, 2048)
Test: (1944, 2048)


In [5]:

import numpy as np
import pandas as pd
import json
import os

from itertools import product
from xgboost import XGBClassifier
from sklearn.metrics import (
    confusion_matrix,
    matthews_corrcoef,
    average_precision_score,
    roc_auc_score
)

# Assumes X_train, X_val, y_train, y_val already exist.
# Do not use X_test or y_test during fine-tuning.

# -----------------------------------------
# 1. Define a modest hyperparameter search
# -----------------------------------------
param_grid = list(product(
    [3, 4, 5],       # max_depth
    [1, 3],          # min_child_weight
    [0.03, 0.05],    # learning_rate
    [0.8],           # subsample
    [0.8],           # colsample_bytree
    [1.0, 1.5]       # scale_pos_weight
))

# Deterministically sample 15 combinations
rng = np.random.default_rng(123)
selected_indices = rng.choice(
    len(param_grid), size=15, replace=False
)
selected_params = [param_grid[i] for i in selected_indices]

# -----------------------------------------
# 2. Find the best threshold for >=90%
#    sensitivity on validation data
# -----------------------------------------
def choose_threshold(y_true, probabilities, target_sensitivity=0.90):
    candidates = np.unique(
        np.concatenate(([0.0], probabilities, [1.0]))
    )

    best = None

    for threshold in candidates:
        predictions = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_true, predictions, labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        mcc = matthews_corrcoef(y_true, predictions)

        if sensitivity >= target_sensitivity:
            candidate = {
                "threshold": float(threshold),
                "sensitivity": float(sensitivity),
                "specificity": float(specificity),
                "MCC": float(mcc),
                "TN": int(tn),
                "FP": int(fp),
                "FN": int(fn),
                "TP": int(tp)
            }

            # Maximize MCC; use specificity to break ties.
            if (
                best is None
                or candidate["MCC"] > best["MCC"]
                or (
                    np.isclose(candidate["MCC"], best["MCC"])
                    and candidate["specificity"] > best["specificity"]
                )
            ):
                best = candidate

    return best

# -----------------------------------------
# 3. Train and evaluate each configuration
# -----------------------------------------
results = []
trained_models = []

for i, params in enumerate(selected_params, start=1):
    (
        max_depth,
        min_child_weight,
        learning_rate,
        subsample,
        colsample_bytree,
        scale_pos_weight
    ) = params

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=600,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        scale_pos_weight=scale_pos_weight,
        reg_lambda=2.0,
        random_state=123,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(X_train, y_train)

    val_prob = model.predict_proba(X_val)[:, 1]

    operating_point = choose_threshold(
        y_val, val_prob, target_sensitivity=0.90
    )

    result = {
        "trial": i,
        "max_depth": max_depth,
        "min_child_weight": min_child_weight,
        "learning_rate": learning_rate,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "scale_pos_weight": scale_pos_weight,
        "val_PR_AUC": average_precision_score(y_val, val_prob),
        "val_ROC_AUC": roc_auc_score(y_val, val_prob),
        **(operating_point or {
            "threshold": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "MCC": -1.0,
            "TN": 0, "FP": 0, "FN": 0, "TP": 0
        })
    }

    results.append(result)
    trained_models.append((model, val_prob, result))

    print(
        f"Trial {i:02d}/15 | "
        f"PR-AUC={result['val_PR_AUC']:.4f} | "
        f"MCC@90% sensitivity={result['MCC']:.4f} | "
        f"Sensitivity={result['sensitivity']:.4f}"
    )

# -----------------------------------------
# 4. Rank models by validation MCC
#    subject to >=90% sensitivity
# -----------------------------------------
results_df = pd.DataFrame(results).sort_values(
    ["MCC", "val_PR_AUC", "specificity"],
    ascending=[False, False, False]
).reset_index(drop=True)

display(results_df)

valid_results = results_df[results_df["sensitivity"] >= 0.90]

if valid_results.empty:
    print(
        "No configuration met 90% validation sensitivity. "
        "Inspect the results before relaxing the target."
    )
else:
    best_trial = int(valid_results.iloc[0]["trial"])
    best_model, best_val_prob, best_result = trained_models[best_trial - 1]

    print("\nBEST CONFIGURATION")
    print(best_result)

    # Save model and validation predictions
    os.makedirs("/kaggle/working/herg_results", exist_ok=True)

    best_model.save_model(
        "/kaggle/working/herg_results/xgboost_tuned.json"
    )

    pd.DataFrame({
        "y_true": y_val,
        "probability": best_val_prob
    }).to_csv(
        "/kaggle/working/herg_results/xgboost_tuned_val.csv",
        index=False
    )

    results_df.to_csv(
        "/kaggle/working/herg_results/xgboost_tuning_results.csv",
        index=False
    )

    with open(
        "/kaggle/working/herg_results/xgboost_tuned_threshold.json",
        "w"
    ) as f:
        json.dump({
            "threshold": best_result["threshold"],
            "target_sensitivity": 0.90,
            "validation_sensitivity": best_result["sensitivity"],
            "validation_MCC": best_result["MCC"],
            "parameters": best_model.get_params()
        }, f, indent=2)

    print("\nSaved tuned model, validation probabilities, and search results.")


Trial 01/15 | PR-AUC=0.8903 | MCC@90% sensitivity=0.3985 | Sensitivity=0.9261
Trial 02/15 | PR-AUC=0.9020 | MCC@90% sensitivity=0.4380 | Sensitivity=0.9035
Trial 03/15 | PR-AUC=0.9011 | MCC@90% sensitivity=0.4005 | Sensitivity=0.9201
Trial 04/15 | PR-AUC=0.8987 | MCC@90% sensitivity=0.4300 | Sensitivity=0.9005
Trial 05/15 | PR-AUC=0.8966 | MCC@90% sensitivity=0.4020 | Sensitivity=0.9005
Trial 06/15 | PR-AUC=0.8920 | MCC@90% sensitivity=0.3632 | Sensitivity=0.9080
Trial 07/15 | PR-AUC=0.8978 | MCC@90% sensitivity=0.3897 | Sensitivity=0.9065
Trial 08/15 | PR-AUC=0.9037 | MCC@90% sensitivity=0.4263 | Sensitivity=0.9186
Trial 09/15 | PR-AUC=0.9006 | MCC@90% sensitivity=0.4250 | Sensitivity=0.9050
Trial 10/15 | PR-AUC=0.8915 | MCC@90% sensitivity=0.3988 | Sensitivity=0.9005
Trial 11/15 | PR-AUC=0.9085 | MCC@90% sensitivity=0.4227 | Sensitivity=0.9110
Trial 12/15 | PR-AUC=0.8965 | MCC@90% sensitivity=0.4182 | Sensitivity=0.9065
Trial 13/15 | PR-AUC=0.9084 | MCC@90% sensitivity=0.4219 | Sensi

,trial,max_depth,min_child_weight,learning_rate,subsample,colsample_bytree,scale_pos_weight,val_PR_AUC,val_ROC_AUC,threshold,sensitivity,specificity,MCC,TN,FP,FN,TP
0,2,5,1,0.03,0.8,0.8,1.0,0.902035,0.809877,0.526437,0.903469,0.487973,0.438012,142,149,64,599
1,14,5,1,0.03,0.8,0.8,1.5,0.902717,0.807742,0.606865,0.906486,0.477663,0.433718,139,152,62,601
2,4,4,3,0.03,0.8,0.8,1.0,0.898728,0.805868,0.538352,0.900452,0.484536,0.430037,141,150,66,597
3,8,4,1,0.05,0.8,0.8,1.5,0.903684,0.806863,0.585200,0.918552,0.446735,0.426269,130,161,54,609
4,9,4,1,0.05,0.8,0.8,1.0,0.900605,0.803188,0.503007,0.904977,0.470790,0.425032,137,154,63,600
5,11,5,1,0.05,0.8,0.8,1.0,0.908520,0.813907,0.491542,0.911011,0.457045,0.422657,133,158,59,604
6,13,5,1,0.05,0.8,0.8,1.5,0.908389,0.814863,0.571649,0.904977,0.467354,0.421923,136,155,63,600
7,12,4,1,0.03,0.8,0.8,1.5,0.896542,0.801403,0.618259,0.906486,0.460481,0.418184,134,157,62,601
8,15,4,3,0.03,0.8,0.8,1.5,0.896974,0.802921,0.608030,0.917044,0.436426,0.414201,127,164,55,608
9,5,3,3,0.05,0.8,0.8,1.0,0.896593,0.800721,0.517192,0.900452,0.453608,0.401964,132,159,66,597



BEST CONFIGURATION
{'trial': 2, 'max_depth': 5, 'min_child_weight': 1, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8, 'scale_pos_weight': 1.0, 'val_PR_AUC': np.float64(0.9020349343079967), 'val_ROC_AUC': np.float64(0.8098770039340082), 'threshold': 0.5264371633529663, 'sensitivity': 0.9034690799396682, 'specificity': 0.4879725085910653, 'MCC': 0.4380118042873328, 'TN': 142, 'FP': 149, 'FN': 64, 'TP': 599}

Saved tuned model, validation probabilities, and search results.


In [6]:
# xgb_model = XGBClassifier(
#     objective="binary:logistic",
#     eval_metric="aucpr",
#     n_estimators=600,
#     learning_rate=0.04,
#     max_depth=5,
#     min_child_weight=3,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_lambda=2.0,
#     random_state=123,
#     n_jobs=-1,
#     tree_method="hist"
# )

# xgb_model.fit(X_train, y_train)

# # Probabilities for class 1 = hERG blocker
# xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]
# xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]

# print("Training complete.")
# print("Validation predictions:", xgb_val_prob.shape)
# print("Test predictions:", xgb_test_prob.shape)

In [7]:
# from sklearn.metrics import confusion_matrix, matthews_corrcoef

# threshold_results = []

# for threshold in np.unique(xgb_val_prob):
#     pred = (xgb_val_prob >= threshold).astype(int)

#     tn, fp, fn, tp = confusion_matrix(
#         y_val, pred, labels=[0, 1]
#     ).ravel()

#     threshold_results.append({
#         "threshold": float(threshold),
#         "sensitivity": tp / (tp + fn),
#         "specificity": tn / (tn + fp),
#         "MCC": matthews_corrcoef(y_val, pred)
#     })

# threshold_df = pd.DataFrame(threshold_results)

# # A: maximum MCC
# best_mcc_row = threshold_df.loc[threshold_df["MCC"].idxmax()]
# xgb_threshold_mcc = float(best_mcc_row["threshold"])

# # B: maximum specificity while sensitivity >= 90%
# eligible = threshold_df[
#     threshold_df["sensitivity"] >= 0.90
# ]

# if eligible.empty:
#     xgb_threshold_sens = None
# else:
#     best_sens_row = eligible.sort_values(
#         ["specificity", "threshold"],
#         ascending=[False, False]
#     ).iloc[0]

#     xgb_threshold_sens = float(best_sens_row["threshold"])

# print("Maximum MCC threshold:")
# print(best_mcc_row.to_string())

# print("\n90% sensitivity threshold:")
# if xgb_threshold_sens is not None:
#     print(best_sens_row.to_string())
# else:
#     print("No threshold achieves 90% validation sensitivity.")

In [8]:

from sklearn.metrics import confusion_matrix, matthews_corrcoef
import numpy as np
import pandas as pd

# Use probabilities from the best tuned model
tuned_val_prob = best_val_prob

threshold_results = []

for threshold in np.unique(
    np.concatenate(([0.0], tuned_val_prob, [1.0]))
):
    pred = (tuned_val_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, pred, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "threshold": float(threshold),
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "MCC": matthews_corrcoef(y_val, pred)
    })

threshold_df = pd.DataFrame(threshold_results)

# A: Maximum MCC, regardless of sensitivity
best_mcc_row = threshold_df.loc[
    threshold_df["MCC"].idxmax()
]
xgb_tuned_threshold_mcc = float(best_mcc_row["threshold"])

# B: Maximum MCC subject to >=90% sensitivity
eligible = threshold_df[
    threshold_df["sensitivity"] >= 0.90
]

if eligible.empty:
    xgb_tuned_threshold_90 = None
    best_90_row = None
else:
    best_90_row = eligible.sort_values(
        ["MCC", "specificity"],
        ascending=[False, False]
    ).iloc[0]

    xgb_tuned_threshold_90 = float(
        best_90_row["threshold"]
    )

print("Tuned XGBoost — maximum MCC:")
print(best_mcc_row.to_string())

print("\nTuned XGBoost — maximum MCC at >=90% sensitivity:")
if best_90_row is not None:
    print(best_90_row.to_string())
else:
    print("No threshold achieves 90% validation sensitivity.")


Tuned XGBoost — maximum MCC:
threshold      0.546333
sensitivity    0.885370
specificity    0.529210
MCC            0.446249

Tuned XGBoost — maximum MCC at >=90% sensitivity:
threshold      0.526437
sensitivity    0.903469
specificity    0.487973
MCC            0.438012


In [9]:
# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     roc_auc_score,
#     average_precision_score,
#     matthews_corrcoef
# )

# def evaluate_model(name, y_true, probabilities, threshold):
#     predictions = (probabilities >= threshold).astype(int)

#     tn, fp, fn, tp = confusion_matrix(
#         y_true, predictions, labels=[0, 1]
#     ).ravel()

#     result = {
#         "Model/Strategy": name,
#         "Threshold": threshold,
#         "Accuracy": accuracy_score(y_true, predictions),
#         "Precision": precision_score(
#             y_true, predictions, zero_division=0
#         ),
#         "Sensitivity": recall_score(
#             y_true, predictions, zero_division=0
#         ),
#         "Specificity": tn / (tn + fp),
#         "F1": f1_score(y_true, predictions, zero_division=0),
#         "ROC-AUC": roc_auc_score(y_true, probabilities),
#         "PR-AUC": average_precision_score(y_true, probabilities),
#         "MCC": matthews_corrcoef(y_true, predictions),
#         "TN": tn, "FP": fp, "FN": fn, "TP": tp
#     }

#     return result


# xgb_test_results = []

# xgb_test_results.append(
#     evaluate_model(
#         "XGBoost — max MCC",
#         y_test, xgb_test_prob, xgb_threshold_mcc
#     )
# )

# if xgb_threshold_sens is not None:
#     xgb_test_results.append(
#         evaluate_model(
#             "XGBoost — 90% sensitivity",
#             y_test, xgb_test_prob, xgb_threshold_sens
#         )
#     )

# xgb_results_df = pd.DataFrame(xgb_test_results)
# display(xgb_results_df.round(4))

# # Save predictions for later comparison and ensembling
# os.makedirs("/kaggle/working/herg_results", exist_ok=True)

# pd.DataFrame({
#     "smiles": test_df["smiles"],
#     "y_true": y_test,
#     "xgboost_probability": xgb_test_prob
# }).to_csv(
#     "/kaggle/working/herg_results/xgboost_test_predictions.csv",
#     index=False
# )

# pd.DataFrame({
#     "smiles": val_df["smiles"],
#     "y_true": y_val,
#     "xgboost_probability": xgb_val_prob
# }).to_csv(
#     "/kaggle/working/herg_results/xgboost_val_predictions.csv",
#     index=False
# )

# xgb_model.save_model(
#     "/kaggle/working/herg_results/xgboost_herg.json"
# )

# print("Model and predictions saved.")

In [10]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)
import pandas as pd
import numpy as np
import os

# -----------------------------------------
# 1. Evaluate a model at a fixed threshold
# -----------------------------------------
def evaluate_model(name, y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        "Model/Strategy": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Sensitivity": recall_score(
            y_true, predictions, zero_division=0
        ),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities),
        "MCC": matthews_corrcoef(y_true, predictions),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


# -----------------------------------------
# 2. Generate test probabilities using the
#    best model found during fine-tuning
# -----------------------------------------
xgb_tuned_test_prob = best_model.predict_proba(X_test)[:, 1]

xgb_test_results = []

# Strategy A: best unconstrained MCC threshold
xgb_test_results.append(
    evaluate_model(
        "Tuned XGBoost — max MCC",
        y_test,
        xgb_tuned_test_prob,
        xgb_tuned_threshold_mcc
    )
)

# Strategy B: best MCC among thresholds with
# at least 90% validation sensitivity
if xgb_tuned_threshold_90 is not None:
    xgb_test_results.append(
        evaluate_model(
            "Tuned XGBoost — MCC at >=90% sensitivity",
            y_test,
            xgb_tuned_test_prob,
            xgb_tuned_threshold_90
        )
    )

xgb_results_df = pd.DataFrame(xgb_test_results)
display(xgb_results_df.round(4))


# -----------------------------------------
# 3. Save validation and test predictions
#    for later comparison and ensembling
# -----------------------------------------
os.makedirs("/kaggle/working/herg_results", exist_ok=True)

pd.DataFrame({
    "smiles": val_df["smiles"].to_numpy(),
    "y_true": y_val,
    "xgboost_probability": best_val_prob
}).to_csv(
    "/kaggle/working/herg_results/xgboost_tuned_val_predictions.csv",
    index=False
)

pd.DataFrame({
    "smiles": test_df["smiles"].to_numpy(),
    "y_true": y_test,
    "xgboost_probability": xgb_tuned_test_prob
}).to_csv(
    "/kaggle/working/herg_results/xgboost_tuned_test_predictions.csv",
    index=False
)

# Save the selected tuned model
best_model.save_model(
    "/kaggle/working/herg_results/xgboost_tuned.json"
)

xgb_results_df.to_csv(
    "/kaggle/working/herg_results/xgboost_tuned_test_metrics.csv",
    index=False
)

print("Tuned model and predictions saved.")


,Model/Strategy,Threshold,Accuracy,Precision,Sensitivity,Specificity,F1,ROC-AUC,PR-AUC,MCC,TN,FP,FN,TP
0,Tuned XGBoost — max MCC,0.5463,0.7495,0.7880,0.8668,0.4959,0.8255,0.7682,0.8713,0.3907,305,310,177,1152
1,Tuned XGBoost — MCC at >=90% sensitivity,0.5264,0.7577,0.7849,0.8894,0.4732,0.8339,0.7682,0.8713,0.4036,291,324,147,1182


Tuned model and predictions saved.
